In [ ]:
###
#SPEI-SR FROM TERRACLIMATE
#Custom index used in the DDWW manuscript:

 #   SR = P - AET - Q

#where
#   P   = TerraClimate precipitation
#  AET = TerraClimate actual evapotranspiration
# Q   = TerraClimate runoff

#For each accumulation timescale k:

#SR_k(t) = sum[SR(t-j), j=0,...,k-1]

#Then:
#1. Fit a 3-parameter log-logistic distribution
# 2. Fit SEPARATELY for each calendar month
#3. Calibration/reference period = 1979-2023
#4. Transform fitted CDF to standard-normal values

#This is a custom SPEI-like index based on P-AET-Q.

#It is NOT conventional SPEI, because conventional SPEI uses P-PET.

#P-AET-Q is interpreted as a water-balance/storage-change residual,
#NOT absolute terrestrial water storage.

#Native TerraClimate (~1/24 degree) is first aggregated to 0.5 degree
#to make the global calculation computationally manageable.
###

from pathlib import Path
import gc
import warnings

import numpy as np
import xarray as xr
from scipy.special import gamma
from scipy.stats import norm
from netCDF4 import Dataset

warnings.filterwarnings("ignore")

# 1. INPUTS

PPT_FP = Path(
    r"F:\Historical data\PP\TerraClimate_ppt_1958_2023_monthly.nc"
)

AET_FP = Path(
    r"F:\Historical data\AET\TerraClimate_aet_1958_2023_monthly.nc"
)

Q_FP = Path(
    r"F:\Historical data\Q\TerraClimate_q_1958_2023_monthly.nc"
)

OUT_DIR = Path(
    r"F:\Historical data\Derived\SPEI_SR"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

SR_FP = (
    OUT_DIR /
    "TerraClimate_SR_P_AET_Q_0p5deg_1958_2023.nc"
)

FINAL_FP = (
    OUT_DIR /
    "TerraClimate_SPEI_SR_0p5deg_1979_2023.nc"
)

# Native TerraClimate = 1/24 degree.
# 12 native pixels = 0.5 degree.
COARSEN_FACTOR = 12

# Explicit source period
SOURCE_START = "1958-01-01"
SOURCE_END   = "2023-12-31"

# Fixed calibration period
CAL_START = "1979-01-01"
CAL_END   = "2023-12-31"

# Period written to final file
OUTPUT_START = "1979-01-01"
OUTPUT_END   = "2023-12-31"

# SPEI-SR accumulation scales
TIMESCALES = [1, 3, 6, 9, 12]

# Minimum number of calibration years required
# 1979-2023 gives 45 observations per calendar month.
MIN_CAL_YEARS = 30

# Prevent norm.ppf(0) / norm.ppf(1)
PROB_EPS = 1.0e-6

# Spatial tile size used during standardization.
# 30 latitude rows at 0.5 degree is memory-safe.
LAT_TILE = 30

# Compression
COMPRESSION_LEVEL = 4
print("=" * 80)
print("CHECKING INPUT FILES")
print("=" * 80)

for fp in [PPT_FP, AET_FP, Q_FP]:

    if not fp.exists():
        raise FileNotFoundError(f"Missing file:\n{fp}")

    print("FOUND:", fp)

print()

# Chunks.


chunks = {
    "time": 12,
    "lat": 720,
    "lon": 1440,
}

print("=" * 80)
print("OPENING TERRACLIMATE DATA")
print("=" * 80)

ds_p = xr.open_dataset(
    PPT_FP,
    decode_times=True,
    mask_and_scale=True,
    chunks=chunks,
)

ds_a = xr.open_dataset(
    AET_FP,
    decode_times=True,
    mask_and_scale=True,
    chunks=chunks,
)

ds_q = xr.open_dataset(
    Q_FP,
    decode_times=True,
    mask_and_scale=True,
    chunks=chunks,
)

if "ppt" not in ds_p:
    raise KeyError(
        f"'ppt' not found in precipitation file.\n"
        f"Variables: {list(ds_p.data_vars)}"
    )

if "aet" not in ds_a:
    raise KeyError(
        f"'aet' not found in AET file.\n"
        f"Variables: {list(ds_a.data_vars)}"
    )

if "q" not in ds_q:
    raise KeyError(
        f"'q' not found in runoff file.\n"
        f"Variables: {list(ds_q.data_vars)}"
    )


P = ds_p["ppt"]
AET = ds_a["aet"]
Q = ds_q["q"]

print("\nREAL DECODED FILE COVERAGE")
print("-" * 80)

for name, da in [
    ("PPT", P),
    ("AET", AET),
    ("Q", Q),
]:

    print(
        f"{name:5s}: "
        f"{str(da.time.values[0])[:10]} to "
        f"{str(da.time.values[-1])[:10]} "
        f"| N = {da.sizes['time']}"
    )

print(
    "\nNOTE: 1958-01 through 2023-12 should contain exactly "
    "792 monthly observations."
)
P = P.sel(
    time=slice(SOURCE_START, SOURCE_END)
)

AET = AET.sel(
    time=slice(SOURCE_START, SOURCE_END)
)

Q = Q.sel(
    time=slice(SOURCE_START, SOURCE_END)
)

print("\nAligning P, AET and Q...")

P, AET, Q = xr.align(
    P,
    AET,
    Q,
    join="exact",
)

ntime = P.sizes["time"]

print(
    f"Selected period: "
    f"{str(P.time.values[0])[:10]} to "
    f"{str(P.time.values[-1])[:10]}"
)

print("Number of months:", ntime)

if ntime != 792:
    raise RuntimeError(
        "\nUnexpected number of months after selecting "
        "1958-01 to 2023-12.\n"
        f"Expected: 792\n"
        f"Found   : {ntime}\n\n"
        "STOPPING so that the time axis can be checked."
    )

print("Time-axis check PASSED.")

if not np.array_equal(
    P["lat"].values,
    AET["lat"].values
):
    raise RuntimeError(
        "P and AET latitude grids differ."
    )

if not np.array_equal(
    P["lat"].values,
    Q["lat"].values
):
    raise RuntimeError(
        "P and Q latitude grids differ."
    )

if not np.array_equal(
    P["lon"].values,
    AET["lon"].values
):
    raise RuntimeError(
        "P and AET longitude grids differ."
    )

if not np.array_equal(
    P["lon"].values,
    Q["lon"].values
):
    raise RuntimeError(
        "P and Q longitude grids differ."
    )

print("\nGRID CHECK PASSED")

print(
    "Native grid:",
    P.sizes["lat"],
    "x",
    P.sizes["lon"],
)

print(
    "Native resolution approximately:",
    abs(float(P.lat[1] - P.lat[0])),
    "x",
    abs(float(P.lon[1] - P.lon[0])),
    "degrees"
)

print("\nVARIABLE UNITS")
print("-" * 80)

print(
    "PPT:",
    P.attrs.get("units", "UNKNOWN")
)

print(
    "AET:",
    AET.attrs.get("units", "UNKNOWN")
)

print(
    "Q  :",
    Q.attrs.get("units", "UNKNOWN")
)

for name, da in [
    ("PPT", P),
    ("AET", AET),
    ("Q", Q),
]:

    units = str(
        da.attrs.get(
            "units",
            ""
        )
    ).lower()

    if "mm" not in units:

        raise RuntimeError(
            f"{name} does not appear to be in mm.\n"
            f"Reported units: {da.attrs.get('units')}"
        )

print("Unit check PASSED.")

print("\nRunning small range check...")

qc_period = slice(
    "2000-01-01",
    "2000-12-31"
)

for name, da in [
    ("PPT", P.sel(time=qc_period)),
    ("AET", AET.sel(time=qc_period)),
    ("Q", Q.sel(time=qc_period)),
]:

    vmin = float(
        da.min(
            skipna=True
        ).compute()
    )

    vmax = float(
        da.max(
            skipna=True
        ).compute()
    )

    print(
        f"{name:5s}: "
        f"min={vmin:.4f}, "
        f"max={vmax:.4f}"
    )


# MONTHLY STORAGE RESIDUAL


print("\n" + "=" * 80)
print("CALCULATING MONTHLY SR = P - AET - Q")
print("=" * 80)

# Only calculate where all three variables exist.
valid = (
    P.notnull()
    & AET.notnull()
    & Q.notnull()
)

SR_native = xr.where(
    valid,
    P - AET - Q,
    np.nan,
)

SR_native = SR_native.astype(
    "float32"
)

SR_native.name = "SR"

SR_native.attrs = {
    "long_name":
        "Monthly water-balance storage residual",
    "definition":
        "SR = precipitation - actual evapotranspiration - runoff",
    "formula":
        "SR = P - AET - Q",
    "units":
        "mm",
    "source":
        "TerraClimate",
    "interpretation":
        "Monthly water-balance/storage-change residual",
    "warning":
        "SR is not absolute terrestrial water storage.",
}


print(
    "\nAggregating native TerraClimate "
    f"{COARSEN_FACTOR} x {COARSEN_FACTOR} cells..."
)

# Since SR is a depth quantity in mm,

SR_coarse = (
    SR_native
    .coarsen(
        lat=COARSEN_FACTOR,
        lon=COARSEN_FACTOR,
        boundary="trim",
    )
    .mean(
        skipna=True
    )
    .astype("float32")
)


N_VALID = (
    valid
    .astype("int16")
    .coarsen(
        lat=COARSEN_FACTOR,
        lon=COARSEN_FACTOR,
        boundary="trim",
    )
    .sum()
    .astype("int16")
)

print(
    "Coarse grid:",
    SR_coarse.sizes["lat"],
    "x",
    SR_coarse.sizes["lon"],
)

print(
    "Coarse resolution approximately:",
    abs(
        float(
            SR_coarse.lat[1]
            - SR_coarse.lat[0]
        )
    ),
    "x",
    abs(
        float(
            SR_coarse.lon[1]
            - SR_coarse.lon[0]
        )
    ),
    "degrees"
)


#  SAVE MONTHLY SR


print("\nSaving coarse SR file:")
print(SR_FP)

sr_out = xr.Dataset(
    {
        "SR": SR_coarse,
        "n_valid_native_cells":
            N_VALID,
    }
)

sr_out.attrs = {
    "title":
        "TerraClimate monthly water-balance storage residual",
    "formula":
        "SR = P - AET - Q",
    "source":
        "TerraClimate",
    "input_period":
        "1958-01 to 2023-12",
    "native_resolution":
        "1/24 degree",
    "output_resolution":
        "approximately 0.5 degree",
    "spatial_aggregation":
        "mean of available native land cells",
    "warning":
        "SR represents storage-change residual, "
        "not absolute storage state.",
}

encoding_sr = {
    "SR": {
        "dtype": "float32",
        "zlib": True,
        "complevel":
            COMPRESSION_LEVEL,
        "_FillValue":
            np.float32(-9999.0),
    },
    "n_valid_native_cells": {
        "dtype": "int16",
        "zlib": True,
        "complevel":
            COMPRESSION_LEVEL,
        "_FillValue":
            np.int16(-9999),
    },
}

sr_out.to_netcdf(
    SR_FP,
    mode="w",
    engine="netcdf4",
    encoding=encoding_sr,
)

print("SR file successfully created.")

ds_p.close()
ds_a.close()
ds_q.close()

del P
del AET
del Q
del SR_native
del SR_coarse
del N_VALID
del valid
del sr_out

gc.collect()


print("\n" + "=" * 80)
print("OPENING COARSE SR FOR STANDARDIZATION")
print("=" * 80)

ds_sr = xr.open_dataset(
    SR_FP,
    decode_times=True,
)

sr = ds_sr["SR"].transpose(
    "time",
    "lat",
    "lon",
)

times = sr["time"].values
lat = sr["lat"].values
lon = sr["lon"].values

NT = len(times)
NY = len(lat)
NX = len(lon)

print("SR dimensions:")
print("time =", NT)
print("lat  =", NY)
print("lon  =", NX)



dates_month = (
    times.astype("datetime64[M]")
)

years = (
    dates_month
    .astype("datetime64[Y]")
    .astype(int)
    + 1970
)

months = (
    dates_month.astype(int)
    % 12
    + 1
)

cal_y0 = int(
    CAL_START[:4]
)

cal_y1 = int(
    CAL_END[:4]
)

out_y0 = int(
    OUTPUT_START[:4]
)

out_y1 = int(
    OUTPUT_END[:4]
)

calibration_mask = (
    (years >= cal_y0)
    & (years <= cal_y1)
)

output_mask = (
    (years >= out_y0)
    & (years <= out_y1)
)

output_indices = np.where(
    output_mask
)[0]

time_out = times[
    output_indices
]

NOUT = len(time_out)

print(
    "\nCalibration period:",
    CAL_START,
    "to",
    CAL_END,
)

print(
    "Output period:",
    OUTPUT_START,
    "to",
    OUTPUT_END,
)

print(
    "Output months:",
    NOUT,
)

if NOUT != 540:

    raise RuntimeError(
        f"Expected 540 months for 1979-2023, found {NOUT}"
    )


def strict_rolling_sum(
    x,
    scale,
):
    """
    Calculate rolling accumulated SR.

    x dimensions:
        [time, lat, lon]

    Requires all months in the accumulation
    window to be finite.

    Partial windows are NOT allowed.
    """

    x = np.asarray(
        x,
        dtype=np.float32,
    )

    nt = x.shape[0]

    out = np.full(
        x.shape,
        np.nan,
        dtype=np.float32,
    )

    if scale == 1:
        return x.copy()

    # Cumulative sum, with NaNs temporarily replaced by zero
    finite = np.isfinite(x)

    x_zero = np.where(
        finite,
        x,
        0.0,
    ).astype(
        np.float32
    )

    cs = np.cumsum(
        x_zero,
        axis=0,
        dtype=np.float64,
    )

    cc = np.cumsum(
        finite.astype(np.int16),
        axis=0,
        dtype=np.int32,
    )

    # first complete window
    out[
        scale - 1
    ] = np.where(
        cc[scale - 1] == scale,
        cs[scale - 1],
        np.nan,
    )

    if nt > scale:

        rolling_sum = (
            cs[scale:]
            - cs[:-scale]
        )

        rolling_count = (
            cc[scale:]
            - cc[:-scale]
        )

        out[scale:] = np.where(
            rolling_count == scale,
            rolling_sum,
            np.nan,
        ).astype(
            np.float32
        )

    return out



# LOG-LOGISTIC 


def fit_loglogistic_pwm(
    x_cal,
    min_n=30,
):
    """
    Fit 3-parameter log-logistic distribution using
    probability weighted moments (PWM).

    Input:
        x_cal = [years, pixels]

    Returns:
        alpha
        beta
        gamma_location

    Invalid fits return NaN.
    """

    x_cal = np.asarray(
        x_cal,
        dtype=np.float64,
    )

    n_pix = x_cal.shape[1]

    alpha = np.full(
        n_pix,
        np.nan,
        dtype=np.float64,
    )

    beta = np.full(
        n_pix,
        np.nan,
        dtype=np.float64,
    )

    gamma_loc = np.full(
        n_pix,
        np.nan,
        dtype=np.float64,
    )

    n_valid = np.sum(
        np.isfinite(x_cal),
        axis=0,
    )

    possible_n = np.unique(
        n_valid[
            n_valid >= min_n
        ]
    )

    for n in possible_n:

        pix = (
            n_valid == n
        )

        if not np.any(pix):
            continue

        xx = x_cal[:, pix]

        # Sort ascending.
        xx_sort = np.sort(
            xx,
            axis=0,
        )[:n, :]

        # Probability plotting position used in SPEI PWM.
        rank = np.arange(
            1,
            n + 1,
            dtype=np.float64,
        )[:, None]

        Fi = (
            rank - 0.35
        ) / float(n)

        w0 = np.mean(
            xx_sort,
            axis=0,
        )

        w1 = np.mean(
            (1.0 - Fi)
            * xx_sort,
            axis=0,
        )

        w2 = np.mean(
            (1.0 - Fi) ** 2
            * xx_sort,
            axis=0,
        )

        numerator = (
            2.0 * w1
            - w0
        )

        denominator = (
            6.0 * w1
            - w0
            - 6.0 * w2
        )

        with np.errstate(
            divide="ignore",
            invalid="ignore",
            over="ignore",
        ):

            b = (
                numerator
                / denominator
            )

        good_b = (
            np.isfinite(b)
            & (b > 1.0)
        )

        if not np.any(
            good_b
        ):
            continue

        G = np.full(
            b.shape,
            np.nan,
            dtype=np.float64,
        )

        G[
            good_b
        ] = (
            gamma(
                1.0
                + 1.0
                / b[good_b]
            )
            *
            gamma(
                1.0
                - 1.0
                / b[good_b]
            )
        )

        a = np.full(
            b.shape,
            np.nan,
            dtype=np.float64,
        )

        with np.errstate(
            divide="ignore",
            invalid="ignore",
            over="ignore",
        ):

            a[
                good_b
            ] = (
                (
                    w0[good_b]
                    - 2.0
                    * w1[good_b]
                )
                *
                b[good_b]
                /
                G[good_b]
            )

        g = np.full(
            b.shape,
            np.nan,
            dtype=np.float64,
        )

        g[
            good_b
        ] = (
            w0[good_b]
            -
            a[good_b]
            *
            G[good_b]
        )

        sample_min = np.nanmin(
            xx,
            axis=0,
        )

        good = (
            good_b
            & np.isfinite(a)
            & (a > 0)
            & np.isfinite(g)
            & (g < sample_min)
        )

        original_indices = np.where(
            pix
        )[0]

        selected = original_indices[
            good
        ]

        alpha[
            selected
        ] = a[
            good
        ]

        beta[
            selected
        ] = b[
            good
        ]

        gamma_loc[
            selected
        ] = g[
            good
        ]

    return (
        alpha,
        beta,
        gamma_loc,
    )



# STANDARD NORMAL


def transform_loglogistic(
    x,
    alpha,
    beta,
    gamma_loc,
    eps=1e-6,
):
    """
    Transform values from fitted
    3-parameter log-logistic CDF
    to a standard-normal variate.
    """

    x = np.asarray(
        x,
        dtype=np.float64,
    )

    alpha = np.asarray(
        alpha,
        dtype=np.float64,
    )

    beta = np.asarray(
        beta,
        dtype=np.float64,
    )

    gamma_loc = np.asarray(
        gamma_loc,
        dtype=np.float64,
    )

    delta = (
        x
        - gamma_loc[None, :]
    )

    valid = (
        np.isfinite(x)
        &
        np.isfinite(
            alpha[None, :]
        )
        &
        np.isfinite(
            beta[None, :]
        )
        &
        np.isfinite(
            gamma_loc[None, :]
        )
        &
        (delta > 0.0)
    )

    F = np.full(
        x.shape,
        np.nan,
        dtype=np.float64,
    )

    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore",
    ):

        ratio = (
            alpha[None, :]
            / delta
        )

        F_all = (
            1.0
            /
            (
                1.0
                +
                ratio **
                beta[None, :]
            )
        )

    F[
        valid
    ] = F_all[
        valid
    ]

    F = np.clip(
        F,
        eps,
        1.0 - eps,
    )

    Z = norm.ppf(
        F
    )

    Z[
        ~valid
    ] = np.nan

    # Equivalent maximum magnitude to probability
    Z = np.clip(
        Z,
        -4.753424,
        4.753424,
    )

    return Z.astype(
        np.float32
    )


def calculate_spei_sr(
    accumulated,
    years,
    months,
    calibration_mask,
):
    """
    accumulated:
        [time, lat, lon]

    Fits each calendar month independently.
    """

    nt, ny, nx = (
        accumulated.shape
    )

    result = np.full(
        accumulated.shape,
        np.nan,
        dtype=np.float32,
    )

    for month in range(
        1,
        13,
    ):

        all_idx = np.where(
            months == month
        )[0]

        cal_idx = np.where(
            (months == month)
            &
            calibration_mask
        )[0]

        x_all = accumulated[
            all_idx
        ].reshape(
            len(all_idx),
            -1,
        )

        x_cal = accumulated[
            cal_idx
        ].reshape(
            len(cal_idx),
            -1,
        )

        (
            alpha,
            beta,
            gamma_loc,
        ) = fit_loglogistic_pwm(
            x_cal,
            min_n=
                MIN_CAL_YEARS,
        )

        z = transform_loglogistic(
            x_all,
            alpha,
            beta,
            gamma_loc,
            eps=PROB_EPS,
        )

        result[
            all_idx
        ] = z.reshape(
            len(all_idx),
            ny,
            nx,
        )

    return result



#  NETCDF


print("\n" + "=" * 80)
print("CREATING FINAL SPEI-SR FILE")
print("=" * 80)

if FINAL_FP.exists():
    FINAL_FP.unlink()

nc = Dataset(
    FINAL_FP,
    "w",
    format="NETCDF4",
)

nc.createDimension(
    "time",
    NOUT,
)

nc.createDimension(
    "lat",
    NY,
)

nc.createDimension(
    "lon",
    NX,
)


# --------------------------------------------------------------
# Coordinates
# --------------------------------------------------------------

vtime = nc.createVariable(
    "time",
    "f8",
    ("time",),
)

vlat = nc.createVariable(
    "lat",
    "f8",
    ("lat",),
)

vlon = nc.createVariable(
    "lon",
    "f8",
    ("lon",),
)


# Use days since 1900 for compatibility.
base_date = np.datetime64(
    "1900-01-01"
)

time_days = (
    time_out.astype(
        "datetime64[D]"
    )
    - base_date
).astype(
    "timedelta64[D]"
).astype(
    np.float64
)

vtime[:] = time_days
vtime.units = (
    "days since 1900-01-01"
)
vtime.calendar = (
    "gregorian"
)
vtime.standard_name = (
    "time"
)

vlat[:] = lat
vlat.units = (
    "degrees_north"
)
vlat.standard_name = (
    "latitude"
)

vlon[:] = lon
vlon.units = (
    "degrees_east"
)
vlon.standard_name = (
    "longitude"
)



output_vars = {}

for scale in TIMESCALES:

    name = (
        f"SPEI_SR_{scale}"
    )

    v = nc.createVariable(
        name,
        "f4",
        (
            "time",
            "lat",
            "lon",
        ),
        zlib=True,
        complevel=
            COMPRESSION_LEVEL,
        fill_value=
            np.float32(
                -9999.0
            ),
        chunksizes=(
            12,
            min(
                LAT_TILE,
                NY,
            ),
            min(
                180,
                NX,
            ),
        ),
    )

    v.long_name = (
        f"{scale}-month standardized "
        "storage-residual index"
    )

    v.units = (
        "standard deviations"
    )

    v.formula = (
        f"standardized rolling {scale}-month "
        "sum of P-AET-Q"
    )

    v.distribution = (
        "three-parameter log-logistic"
    )

    v.parameter_estimation = (
        "probability-weighted moments"
    )

    v.calibration_period = (
        "1979-2023"
    )

    v.calendar_month_fitting = (
        "independent fitting for each "
        "calendar month"
    )

    v.interpretation = (
        "positive values indicate relatively "
        "positive storage-change residual; "
        "negative values indicate relatively "
        "negative storage-change residual"
    )

    v.warning = (
        "This variable represents a standardized "
        "water-balance/storage-change residual, "
        "not absolute water storage."
    )

    output_vars[
        scale
    ] = v



nc.title = (
    "TerraClimate multiscale SPEI-SR"
)

nc.index_name = (
    "SPEI-SR"
)

nc.index_definition = (
    "Custom standardized storage-residual index"
)

nc.monthly_residual = (
    "SR = precipitation - actual evapotranspiration - runoff"
)

nc.input_dataset = (
    "TerraClimate"
)

nc.input_period = (
    "1958-2023"
)

nc.analysis_period = (
    "1979-2023"
)

nc.calibration_period = (
    "1979-2023"
)

nc.timescales = (
    "1,3,6,9,12 months"
)

nc.distribution = (
    "three-parameter log-logistic"
)

nc.parameter_estimation = (
    "probability-weighted moments"
)

nc.calendar_month_standardization = (
    "yes"
)

nc.spatial_resolution = (
    "approximately 0.5 degree"
)

nc.important_note = (
    "P-AET-Q approximates a monthly water-balance/"
    "storage-change residual and must not be "
    "interpreted as absolute terrestrial water storage."
)

nc.sync()


print(
    "\nProcessing spatial tiles..."
)

for lat_start in range(
    0,
    NY,
    LAT_TILE,
):

    lat_end = min(
        lat_start + LAT_TILE,
        NY,
    )

    print(
        "\n" + "-" * 80
    )

    print(
        f"LAT TILE: "
        f"{lat_start}:{lat_end} "
        f"of {NY}"
    )

    # Load only one 0.5-degree latitude tile.
    tile = (
        sr
        .isel(
            lat=slice(
                lat_start,
                lat_end,
            )
        )
        .values
        .astype(
            np.float32
        )
    )

    print(
        "Tile shape:",
        tile.shape,
    )

    if not np.isfinite(
        tile
    ).any():

        print(
            "Tile contains no valid land values. Skipping."
        )

        continue




    for scale in TIMESCALES:

        print(
            f"  SPEI-SR-{scale}: "
            "rolling accumulation..."
        )

        accumulated = (
            strict_rolling_sum(
                tile,
                scale,
            )
        )

        print(
            f"  SPEI-SR-{scale}: "
            "calendar-month standardization..."
        )

        spei_tile = (
            calculate_spei_sr(
                accumulated=
                    accumulated,
                years=years,
                months=months,
                calibration_mask=
                    calibration_mask,
            )
        )

        # Select 1979-2023 only
        spei_out = spei_tile[
            output_indices,
            :,
            :,
        ]

        output_vars[
            scale
        ][
            :,
            lat_start:lat_end,
            :
        ] = spei_out

        nc.sync()

        finite_count = int(
            np.isfinite(
                spei_out
            ).sum()
        )

        print(
            f"  SPEI-SR-{scale}: "
            f"written | "
            f"finite values={finite_count:,}"
        )

        del accumulated
        del spei_tile
        del spei_out

        gc.collect()

    del tile

    gc.collect()




nc.close()
ds_sr.close()

gc.collect()

print("\n" + "=" * 80)
print("SPEI-SR CALCULATION COMPLETED")
print("=" * 80)

print("\nIntermediate SR:")
print(SR_FP)

print("\nFinal SPEI-SR:")
print(FINAL_FP)



print("\n" + "=" * 80)
print("FINAL QUALITY CONTROL")
print("=" * 80)

ds_check = xr.open_dataset(
    FINAL_FP,
    decode_times=True,
)

print(ds_check)

for var in [
    "SPEI_SR_1",
    "SPEI_SR_3",
    "SPEI_SR_6",
    "SPEI_SR_9",
    "SPEI_SR_12",
]:

    da = ds_check[var]

    # Global values are sampled to make QC quick.
    sample_da = da.isel(
        lat=slice(
            None,
            None,
            4,
        ),
        lon=slice(
            None,
            None,
            4,
        ),
    )

    mean_val = float(
        sample_da.mean(
            skipna=True
        ).compute()
    )

    std_val = float(
        sample_da.std(
            skipna=True
        ).compute()
    )

    min_val = float(
        sample_da.min(
            skipna=True
        ).compute()
    )

    max_val = float(
        sample_da.max(
            skipna=True
        ).compute()
    )

    print(
        f"\n{var}"
    )

    print(
        f"  mean = {mean_val:.4f}"
    )

    print(
        f"  std  = {std_val:.4f}"
    )

    print(
        f"  min  = {min_val:.4f}"
    )

    print(
        f"  max  = {max_val:.4f}"
    )


print(
    "\nTime coverage:"
)

print(
    str(
        ds_check.time.values[0]
    )[:10],
    "to",
    str(
        ds_check.time.values[-1]
    )[:10],
)

print(
    "N months =",
    ds_check.sizes["time"],
)

ds_check.close()

print(
    "\nDONE."
)